#Schritt 1: Datensammlung
#Songtexte über die Genius-API abrufen und  als CSV abspeichern. 

#Token: https://genius.com/api-clients

In [ ]:
import pandas as pd
import os

In [ ]:
def lade_daten_mit_genius(song_liste, genius_token, timeout=15, retries=3):
    import lyricsgenius
    genius = lyricsgenius.Genius(genius_token, timeout=timeout, retries=retries)
    genius.remove_section_headers = False  
    ergebnisse = []
    fehlgeschlagen = []
    for artist_name, song_titel in song_liste:
        print(f'Suche: {song_titel} von {artist_name} ...')
        try:
            song = genius.search_song(song_titel, artist_name)
        except Exception as fehler:
            #falls API irgendeinen Fehler zurückgibt
            print(f'Fehler bei diesem Song ({type(fehler).__name__}: {fehler}), wird übersprungen')
            fehlgeschlagen.append((artist_name, song_titel, str(fehler)))
            continue
        if song is not None:
            ergebnisse.append({'artist': artist_name, 'title': song_titel, 'lyrics': song.lyrics})
        else:
            print(f'nicht gefunden, wird übersprungen')
            fehlgeschlagen.append((artist_name, song_titel, 'nicht gefunden'))
    daten = pd.DataFrame(ergebnisse)
    print(f'\n{len(ergebnisse)} von {len(song_liste)} Songs erfolgreich geladen.')
    if fehlgeschlagen:
        print(f'{len(fehlgeschlagen)} Songs übersprungen:')
        for artist_name, song_titel, grund in fehlgeschlagen:
            print(f'  - {song_titel} ({artist_name}): {grund}')
        print('diese Songs ggf.  nochmal probieren,')
    return daten

In [ ]:
def speichere_rohdaten(daten, ausgabe_pfad='data/rohdaten.csv'):
    ordner = os.path.dirname(ausgabe_pfad)
    if ordner and (not os.path.isdir(ordner)):
        os.makedirs(ordner, exist_ok=True)
        print(f'Ordner {ordner} wurde neu angelegt.')
    daten.to_csv(ausgabe_pfad, index=False)
    print(f'Rohdaten gespeichert unter: {ausgabe_pfad}')

In [ ]:
def ergänze_pflichtspalten(daten, playlist_id='P1'):
    daten = daten.copy()
    daten['song_id'] = ['S' + str(i + 1) for i in range(len(daten))]
    daten['playlist_id'] = playlist_id
    return daten

#Pipeline ausführen

In [ ]:
mein_token = 'HIER_GENIUS_TOKEN_EINFUEGEN'
if mein_token == 'HIER_GENIUS_TOKEN_EINFUEGEN':
    raise ValueError('Bitte oben in dieser Zelle den Genius-Token anstelle des Texts eintragen.')
else:
    print('Token eingetragen')

#Songliste

In [ ]:
songs = [('Adele', 'Someone Like You'), ('Billie Eilish', "when the party's over"), ('Sam Smith', 'Stay With Me'), ('Lewis Capaldi', 'Someone You Loved'), ('Olivia Rodrigo', 'drivers license'), ('The Beatles', 'Let It Be'), ('Coldplay', 'Fix You'), ('Pharrell Williams', 'Happy'), ('Katy Perry', 'Firework'), ('Queen', "Don't Stop Me Now"), ('Ed Sheeran', 'Photograph'), ('Adele', 'Someone Like You'), ('Green Day', 'Boulevard of Broken Dreams'), ('Linkin Park', 'Numb'), ('Eminem', 'Lose Yourself'), ('The Weeknd', 'Blinding Lights'), ('Billie Eilish', 'everything i wanted'), ('Radiohead', 'No Surprises'), ('Simon & Garfunkel', 'The Sound of Silence'), ('Frank Sinatra', 'My Way'), ('The Fray', 'How to Save a Life'), ('Bon Iver', 'Skinny Love'), ('A Great Big World', 'Say Something'), ('Johnny Cash', 'Hurt'), ('Gary Jules', 'Mad World'), ('Lord Huron', 'The Night We Met'), ('Sam Smith', 'Too Good at Goodbyes'), ("James Arthur", "Say You Won't Let Go")]

In [ ]:
daten = lade_daten_mit_genius(songs, mein_token)
print(daten.head())

In [ ]:
daten = ergänze_pflichtspalten(daten, playlist_id='P1')
print(daten[['song_id', 'playlist_id', 'artist', 'title']])
speichere_rohdaten(daten)